# M8d · Handling Imbalanced Datasets

**Goal:** derive a binary risk label from the existing `v_device_risk_profile`
(or compute one on the fly) and rebalance the class distribution before
training.

**Why imbalance matters:** Project 1's "high-risk driver" class is rare
(~10–15% of device-months). Training on the raw distribution produces
classifiers that simply predict "low risk" 100% of the time and look 85%
accurate while detecting 0 high-risk drivers.

**Three rebalancing strategies covered:**

| Technique          | Mechanism                                            | When                                    |
|--------------------|------------------------------------------------------|-----------------------------------------|
| Random Undersampling| Drop majority rows                                  | Fast; OK when majority is huge          |
| Random Oversampling | Duplicate minority rows                             | Cheap but causes overfitting             |
| **SMOTE**           | Synthesize new minority samples by interpolation    | The standard; preserves info            |

**Exit criteria:**
- Train/test split made BEFORE rebalancing (golden rule — don't leak SMOTE-generated rows into the test set).
- Class ratio of resampled train set printed and visualized.
- Saved as `(X_train_bal, y_train_bal, X_test, y_test)` parquet quartet.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — encoded features

In [ ]:
import pandas as pd, pathlib, numpy as np
df = pd.read_parquet('../../data/ml/step3_encoded.parquet')
print('shape:', df.shape)


## 3. Derive a binary risk label

Define `is_high_risk = 1` iff this (device, year_month) is in the **top
quartile** by `harsh_events_per_100km` AND has at least 100 km driven that
month (avoids classifying low-distance noise as "high risk").


In [ ]:
ELIGIBLE = df['total_distance_km'] >= 100
threshold = df.loc[ELIGIBLE, 'harsh_events_per_100km'].quantile(0.75)
df['is_high_risk'] = ((df['harsh_events_per_100km'] >= threshold) & ELIGIBLE).astype(int)

print(f'P75 harsh_events_per_100km = {threshold:.2f}')
print(df['is_high_risk'].value_counts(normalize=True).rename('share'))


## 4. Train/test split BEFORE rebalancing

In [ ]:
from sklearn.model_selection import train_test_split

ID_COLS = ['tenant_id', 'device_id', 'year_month']
TARGET  = 'is_high_risk'
DROP    = ID_COLS + [TARGET, 'harsh_events_per_100km',  # leak!
                     'harsh_event_total', 'harsh_brake_count', 'harsh_accel_count',
                     'harsh_corner_count']  # all directly compose the target
X = df.drop(columns=[c for c in DROP if c in df.columns])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('train:', X_train.shape, 'pos rate:', y_train.mean().round(3))
print('test :', X_test.shape,  'pos rate:', y_test.mean().round(3))


## 5. Rebalance with SMOTE

`imblearn.over_sampling.SMOTE` synthesizes minority points by interpolating
between a minority sample and its k-nearest minority neighbours. We
deliberately oversample to a 50/50 ratio for the demo; in practice tune
`sampling_strategy` between 0.3 and 1.0.


In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
print('balanced train:', X_train_bal.shape,
      'pos rate:', y_train_bal.mean().round(3))


## 6. Sanity-check class distributions

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
y_train.value_counts().plot.bar(ax=axes[0], title='Original train')
y_train_bal.value_counts().plot.bar(ax=axes[1], title='SMOTE-balanced train')
plt.tight_layout(); plt.show()


## 7. Persist

In [ ]:
out_dir = pathlib.Path('../../data/ml')
X_train_bal.to_parquet(out_dir / 'X_train_bal.parquet', index=False)
y_train_bal.to_frame('is_high_risk').to_parquet(out_dir / 'y_train_bal.parquet', index=False)
X_test.to_parquet(out_dir / 'X_test.parquet', index=False)
y_test.to_frame('is_high_risk').to_parquet(out_dir / 'y_test.parquet', index=False)
print('train/test artifacts written to', out_dir.resolve())
